# D2 · What `predict_proba` means in a KNN

**Type**: Interpret · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Interpret the `predict_proba` of a KNN: why it takes few values, what effect it has on the log-loss and on the ROC curve, whether it is calibrated, and how a single prediction is "explained" by looking at the neighbors that voted.

## Dataset

`make_classification` (3000 examples, 6 features of which 3 informative), standardized, 50/50 split: `Z_tr, y_tr, Z_te, y_te`. Models (given): KNN with `k` = 1, 5, 25 (uniform), 25 with 1/distance weights, and a logistic as a reference (the problem is linear enough to work well for all).

## Educational purpose

Dismantle an illusion: the `predict_proba` of a KNN **is not a calibrated probability**, it is a fraction of neighbors. It has practical consequences on metrics (huge log-loss with small `k`), curves (stepped ROC) and calibration.

## Task

1. Save `n_values` (dictionary `model name → number of distinct values` in `predict_proba(Z_te)[:, 1]`) for the five models (`knn1`, `knn5`, `knn25`, `knn25_d`, `lr`). How is it linked to `k` for the uniform ones?
2. Save `table` (index: the five models; columns `auc`, `logloss`, `roc_points` (length of `roc_curve(...)[0]`), `ece`) with the `ece` already defined (`ece(y, p)`, 10 groups of equal size).
3. Save `why_logloss_is_high = 'b'`: a) the KNN ranks the cases badly; b) with small `k` many probabilities are exactly 0 or 1 and errors with probability 0/1 cost `-log(1e-15)` ≈ 34.5 each; c) the log-loss cannot be used with KNNs.
4. **Explaining a prediction.** For the first test example that `knn25` gets wrong (`error_idx`, already computed) save `neighbors_table` (DataFrame with one row per neighbor: `index`, `distance`, `class`, sorted by distance) and `votes` (dictionary class → number of votes) using `knn25.kneighbors`. Save `margin` = votes of the predicted class minus votes of the other one.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

X, y = make_classification(3000, n_features=6, n_informative=3, n_redundant=0, n_clusters_per_class=1, class_sep=0.8, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.5, random_state=0)
scaler = StandardScaler().fit(X_tr)
Z_tr, Z_te = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    "knn1": KNeighborsClassifier(1), "knn5": KNeighborsClassifier(5), "knn25": KNeighborsClassifier(25),
    "knn25_d": KNeighborsClassifier(25, weights="distance"), "lr": LogisticRegression(max_iter=1000),
}
probabilities = {name: m.fit(Z_tr, y_tr).predict_proba(Z_te)[:, 1] for name, m in models.items()}
knn25 = models["knn25"]
error_idx = int(np.flatnonzero(knn25.predict(Z_te) != y_te)[0])           # first test example that knn25 gets wrong


def ece(y, p, k=10):
    g = pd.DataFrame({"y": y, "p": p, "g": pd.qcut(p, k, duplicates="drop")}).groupby("g", observed=True).agg(n=("y", "size"), y=("y", "mean"), p=("p", "mean"))
    return float((g["n"] / g["n"].sum() * (g["p"] - g["y"]).abs()).sum())

## Your work

In [ ]:
n_values = None

In [ ]:
table = pd.DataFrame(index=list(models), columns=["auc", "logloss", "roc_points", "ece"], dtype=float)
table

In [ ]:
why_logloss_is_high = None

In [ ]:
neighbors_table = None
votes = None
margin = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Neighbors: `d, idx = knn25.kneighbors(Z_te[[error_idx]])`; `classes = y_tr[idx[0]]`; `votes = {0: int((classes == 0).sum()), 1: int((classes == 1).sum())}`. Log-loss: `log_loss(y_te, p)` (with 0/1 probabilities `scikit-learn` clips them to `1e-15`).'):
    check(n_values["knn1"] == 2 and n_values["knn5"] == 6 and n_values["knn25"] == 26 and n_values["knn25_d"] > 100 and n_values["lr"] > 1000, "k+1 values for the uniform KNNs (2, 6, 26); over a hundred with 1/distance weights; thousands with the logistic", "np.unique(p).size")
    check(table.notna().all().all() and close(table.loc["knn5", "roc_points"], len(roc_curve(y_te, probabilities["knn5"])[0])), "complete table", "five models, four columns")
    check(table.loc["knn1", "logloss"] > 5 and table.loc["knn1", "logloss"] > 5 * table.loc["knn5", "logloss"] and table.loc["knn25", "logloss"] < table.loc["knn5", "logloss"] and table.loc["lr", "logloss"] < table.loc["knn25", "logloss"], "log-loss: over 5 with k=1 (more than 5 times that of k=5), then it drops with k=25 and the logistic is the best", "use sklearn.metrics.log_loss on the raw probabilities")
    check(table.loc["knn1", "roc_points"] == 3 and 10 < table.loc["knn25", "roc_points"] <= 27 and table.loc["lr", "roc_points"] > 50, "the ROC curve has 3 points for k=1, at most 27 for k=25 (scikit-learn drops collinear points), over 50 for the logistic", "len(roc_curve(y, p)[0])")
    check(table.loc["knn25", "auc"] > table.loc["knn1", "auc"] + 0.1 and table.loc["lr", "ece"] < table.loc["knn25", "ece"] / 1.5, "k = 25 has an AUC better than k = 1 by over 0.1; the logistic has an ECE at least 1.5 times better than k = 25", "compare auc and ece")
    check(table.loc["knn1", "ece"] < 0.05, "with k = 1 the ECE is small (below 0.05) but it is an illusion: the groups with equal probabilities collapse into one", "pd.qcut(..., duplicates='drop') with only two distinct values")
    check(why_logloss_is_high == "b", "correct answer: wrong 0/1 probabilities cost 34.5", "b")
    check(list(neighbors_table.columns) == ["index", "distance", "class"] and len(neighbors_table) == 25 and neighbors_table["distance"].is_monotonic_increasing and sum(votes.values()) == 25, "25 neighbors, increasing distances, 25 votes", "knn25.kneighbors(Z_te[[error_idx]])")
    check(margin == max(votes.values()) - min(votes.values()) and int(y_te[error_idx]) != max(votes, key=votes.get), "the margin is the difference in votes; the true class lost the vote", "votes of the predicted class minus votes of the other")
done()

---
The commented solution is in `solutions/D_interpret/D2_knn_probabilities.ipynb`. Open it after you pass the check, or after you have really tried.